In [1]:
from typing import TypedDict, Optional, Dict, Any
from langchain_ollama import ChatOllama
from langchain_experimental.tools import PythonREPLTool
from langchain_core.prompts import PromptTemplate
from langgraph.types import interrupt, Command
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

# 1. State Schema
class DataScienceState(TypedDict):
    dataset_path: str
    dataset_summary: Optional[str]
    missing_values_summary: Optional[str]
    requires_imputation: bool
    imputation_strategy: Optional[str]
    scaling_strategy: Optional[str]
    hyperparameters: Optional[Dict[str, Any]]
    model_metrics: Optional[str] # To store accuracy/loss

# 2. Initialize LLM and REPL
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0)
python_repl = PythonREPLTool()

def extract_code_block(llm_response: str) -> str:
    if "```python" in llm_response:
        return llm_response.split("```python")[1].split("```")[0].strip()
    elif "```" in llm_response:
        return llm_response.split("```")[1].split("```")[0].strip()
    return llm_response.replace("Action: python_repl_ast", "").replace("Action Input:", "").strip()

print("Environment initialized for PyTorch Execution.")

C:\Users\Shivachetan Ulavi\AppData\Local\Temp\ipykernel_11716\1747295660.py:3: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.tools import PythonREPLTool


Environment initialized for PyTorch Execution.


# The Hyperparameter Node HITL

In [2]:
def ask_hyperparameters_node(state: DataScienceState) -> DataScienceState:
    print("-- PAUSED: ASK HYPERPARAMETERS--")
    human_choice = interrupt({
        "question":"Select Training Hyperparameters",
        "options":[
            {"epochs":50, "lr":0.01, "hidden_dim": 32},
            {"epochs":100, "lr":0.005, "hidden_dim": 64},
            {"epochs":200, "lr":0.001, "hidden_dim": 128}]
    })
    print(f"Human Selected: {human_choice}")
    return {"hyperparameters":human_choice}

In [4]:
# (Since California Housing predicts 'MedHouseVal' instead of 'Class')
# Let's temporarily inject the target column into our prompt for this test.

def pytorch_training_node_dynamic(state: DataScienceState, target_col: str = "MedHouseVal") -> DataScienceState:
    print(f"--ENTERING PYTORCH  TRAINING Node (Target:{target_col})--")

    dataset_path = state.get("dataset_path")
    hp = state.get("hyperparameters", {"epochs": 50, "lr": 0.01, "hidden_dim": 64})
    pytorch_prompt = PromptTemplate.from_template(
        """
        You are a PyTorch Expert. A dataset is at '{filepath}'.
        The target column is '{target}'. All other columns are features.
        
        Write a single block of Python code to:
        1. Load the dataset using pandas.
        2. Separate features (X) and target (y), and convert them to PyTorch tensors (torch.float32 for X, torch.float32 for y).
           Ensure y is reshaped to (-1, 1).
        3. Create a PyTorch nn.Module neural network with one hidden layer of size {hidden_dim}.
        4. Initialize device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') and move the model and tensors to this device.
        5. Use MSELoss (since this is continuous) and the Adam optimizer with a learning rate of {lr}.
        6. Train the model for {epochs} epochs.
        7. Print the final epoch's Loss.
        
        CRITICAL INSTRUCTION:
        Return ONLY valid Python code inside a ```python ``` markdown block. Do not explain.
        """
    )

    chain = pytorch_prompt | llm
    response = chain.invoke({
        "filepath": dataset_path,
        "target" : target_col,
        "hidden_dim": hp.get("hidden_dim",64),
        "lr": hp.get("lr", 0.01),
        "epochs": hp.get("epochs", 50)
    })
    executable_code = extract_code_block(response.content)
    print("Generated PyTorch Code:\n", executable_code)

    print("\n Executing PyTorch via REPL...")
    repl_output = python_repl.invoke(executable_code)
    print("REPL Output:\n", repl_output)

    return {"model_metrics": repl_output}

In [5]:
test_state_pytorch: DataScienceState = {
    "dataset_path": "../data/california_housing.csv", 
    "dataset_summary": None,
    "missing_values_summary": None,
    "requires_imputation": False,
    "imputation_strategy": "None",
    "scaling_strategy": "None",
    "hyperparameters": {"epochs": 50, "lr": 0.01, "hidden_dim": 64}, 
    "model_metrics": None
}

updated_state_pytorch = pytorch_training_node_dynamic(test_state_pytorch, target_col="MedHouseVal")
print("\n--- FINAL METRICS ---")
print(updated_state_pytorch["model_metrics"])

--ENTERING PYTORCH  TRAINING Node (Target:MedHouseVal)--


Python REPL can execute arbitrary code. Use with caution.


Generated PyTorch Code:
 import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the dataset
data = pd.read_csv('../data/california_housing.csv')

# Separate features and target
X = data.drop('MedHouseVal', axis=1).values
y = data['MedHouseVal'].values.reshape(-1, 1)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# Create a PyTorch nn.Module neural network
class SimpleNN(nn.Module):
    def __init__(s